In [1]:
# general imports 
import numpy as np
import matplotlib.pyplot as plt
import re
import math

# What is new today?

## MDAnalysis

- Open-source Python library providing tools for analyzing MD simulations.
- Supports multiple file formats, including those from GROMACS (seamlessly reads `.gro`, `.xtc`, `.trr`, `.tpr`, etc.)
- Versatile &mdash; can handle large trajectories and offers flexible functionalities.
- Multiple built-in tools.
- Powerful atom selection.
- Quite intuitive to work with.
- Highly extensible &mdash; usually you write your own advanced analysis based on MDAnalysis skeleton.
- Large community and constant development.
- **We will test it at the very end of today.**

## Useful links

- https://www.mdanalysis.org/
- https://userguide.mdanalysis.org/stable/examples/quickstart.html

# Today's Practical

We will analyze simulations of water/methanol mixtures we prepared (some of them partially, some of them fully) last week. We will check various tools (implemented in GROMACS or available from the third-party resources) and dive into the relation of simulation outputs with available experimental data.

In [ ]:
!ls -l

## What is inside?

### images

- No surprises

### trajectories

Contains simulations (10 ns long each) for five different systems:

- long_pure_water (pure water, 1000 water molecules)
- long_pure_methanol (pure methanol, 1000 methanol molecules)
- long_mix_w3_m1 (mixture 3 to 1, 750 waters and 250 methanols)
- long_mix_w1_m1 (mixture 1 to 1, 500 waters and 500 methanols; our focus last week)
- long_mix_w1_m3 (mixture 1 to 3, 250 waters and 750 methanols)

We also have bigger versions of the systems above:

- long_pure_water_bg (pure water, 2000 water molecules)
- long_pure_water_lg (pure water, 3000 water molecules)
- long_pure_water_hg (pure water, 9000 water molecules)
- long_mix_w1_m1_bg (mixture 1 to 1, 1000 waters and 1000 methanols)
- long_mix_w1_m1_lg (mixture 1 to 1, 1500 waters and 1500 methanols)

### week3.ipynb

Jupyter notebook we are using right now.

## Create a folder where we will be working today

In [ ]:
!mkdir analysis/

In [ ]:
%cd analysis/

In [ ]:
!ls -l

## Analysis using GROMACS tools

## Stability and sanity checks

- Always try to check if your simulations are actually reasonable
- Visual checks are usually very helpful
- Standard checks we did last time (temperature, pressure, box size) are also useful
- Since we have mixtures of two solvents that should be fully mixable, the mixing itself can be also checked

In [ ]:
!echo 'Temperature' | gmx energy -f ../trajectories/long_pure_water.edr -o temp_pure_water.xvg
!echo 'Temperature' | gmx energy -f ../trajectories/long_mix_w3_m1.edr -o temp_mix_w3_m1.xvg
!echo 'Temperature' | gmx energy -f ../trajectories/long_mix_w1_m1.edr -o temp_mix_w1_m1.xvg
!echo 'Temperature' | gmx energy -f ../trajectories/long_mix_w1_m3.edr -o temp_mix_w1_m3.xvg
!echo 'Temperature' | gmx energy -f ../trajectories/long_pure_methanol.edr -o temp_pure_methanol.xvg

### Averages look good

- Try to be smart. We executed the same command on five different systems. Five lines is not that bad. But what if we have 200 systems?
- Using Python for optimization.

In [ ]:
measure = 'Temperature'
systems = ['pure_water', 'mix_w3_m1', 'mix_w1_m1', 'mix_w1_m3', 'pure_methanol']
for system in systems:
    !echo {measure} | gmx energy -f ../trajectories/long_{system}.edr -o temp_{system}.xvg

### Looks okay. Can we do even better?

- We can capture the printed output and extract only the interesting part.

In [ ]:
measure = 'Temperature'
systems = ['pure_water', 'mix_w3_m1', 'mix_w1_m1', 'mix_w1_m3', 'pure_methanol']
for system in systems:
    output =  !echo {measure} | gmx energy -f ../trajectories/long_{system}.edr -o temp_{system}.xvg
    output_str = '\n'.join(output)
    match = re.search(r"Temperature\s+(\d+\.\d+)", output_str)
    if match:
        density = float(match.group(1))
        print(f"Temperature in {system} is {density} K")
    else:
        print("Something went wrong!")

### What did we do?

- We put the output information from the GROMACS command into variable that further was transformed into a string.
- We used Python package `re` that works with regular expressions.
    - Regular expressions are patterns used to match character combinations in strings.
- We looked for the pattern (word) `Temperature` and found the value next to it.
- Then, we printed results on the screen.

### Let's check fluctuations of the temperature

In [2]:
temp_pure_water = np.loadtxt('temp_pure_water.xvg', comments=['#', '@'])
temp_mix_w3_m1 = np.loadtxt('temp_mix_w3_m1.xvg', comments=['#', '@'])
temp_mix_w1_m1 = np.loadtxt('temp_mix_w1_m1.xvg', comments=['#', '@'])
temp_mix_w1_m3 = np.loadtxt('temp_mix_w1_m3.xvg', comments=['#', '@'])
temp_pure_methanol = np.loadtxt('temp_pure_methanol.xvg', comments=['#', '@'])

plt.plot(temp_pure_water[:, 0], temp_pure_water[:, 1], 'r-')
plt.plot(temp_mix_w3_m1[:, 0], temp_mix_w3_m1[:, 1], 'b-')
plt.plot(temp_mix_w1_m1[:, 0], temp_mix_w1_m1[:, 1], 'g-')
plt.plot(temp_mix_w1_m3[:, 0], temp_mix_w1_m3[:, 1], 'k-')
plt.plot(temp_pure_methanol[:, 0], temp_pure_methanol[:, 1], 'y-')

plt.legend(['water', 
            'w3 m1', 
            'w1 m1', 
            'w1 m3', 
            'methanol'])
plt.xlabel('Time (ps)')
plt.ylabel('Temperature (K)')
plt.show()

OSError: temp_pure_water.xvg not found.

### Let's check densities: they should be different for pure water, methanol, mixtures, ...?

- Below the commented commands how to do it one by one.

In [ ]:
# !echo 'Density' | gmx energy -f ../trajectories/long_pure_water.edr -o dens_pure_water.xvg
# !echo 'Density' | gmx energy -f ../trajectories/long_pure_methanol.edr -o dens_pure_methanol.xvg
# !echo 'Density' | gmx energy -f ../trajectories/long_mix_w3_m1.edr -o dens_mix_w3_m1.xvg
# !echo 'Density' | gmx energy -f ../trajectories/long_mix_w1_m1.edr -o dens_mix_w1_m1.xvg
# !echo 'Density' | gmx energy -f ../trajectories/long_mix_w1_m3.edr -o dens_mix_w1_m3.xvg

### TASK

- Again, be smart. Optimize the code and calculate and print densities as we did for temperatures.

### Another check &mdash; mixability

### Defining atom groups

- Most analyses require specifying atom groups.
- `.xtc` trajectories do not know what is water, methanol, part of protein, etc.
- For instance, what if we want to calculate properties of only water when it is in the mixture; only of water oxygens, not hydrogens of water; etc.
- Now we want to check mixability of our mixtures. We need to define corresponding atom groups.

### GROMACS file format we have not covered yet

- `.ndx` &mdash; index file used to define atom groups. These groups can be used at both simulation (thermostatting of different atom groups, Week 5 for more details) and post-processing stages.

### `gmx make_ndx` 

- `gmx make_ndx` command is used to generate and modify `.ndx` files 
- `-f` &mdash; `.gro` structure; no `.gro` structure (i.e., no atoms) &mdash; apparently, no groups to define
- `-n` &mdash; existing `.ndx` file if any
- `-o` &mdash; output `.ndx` file; default name &mdash; `index.ndx`

### Let's create the index file for pure water system as a test

- We will create a new group of water oxygens.

In [ ]:
input_command = """a OW 
q
"""
!echo "$input_command" | gmx make_ndx -f ../trajectories/long_pure_water.gro -o pure_water.ndx

### What did we do?

- `a OW` &mdash; we created a group of atoms that contains atoms named `OW`, i.e., water oxygens (see below)
- `q`  &mdash; save and quit
- there are other ways to define atom groups, see above for tips that pops up upon execution of the command
- note there are predefined groups recognized by GROMACS: `System`, `Water`, `SOL`; typically, GROMACS can elso recognize `Protein`, `Ions`, and other generic groups, but sometimes it works badly

In [ ]:
!head ../trajectories/long_pure_water.gro

### Calculating number density of water oxygens along the box

### `gmx density`

- While `gmx energy` provides the time-dependency of the system mass density, `gmx density` is used to calculate the partial/total densities across the box.
- `-f` &mdash; input trajectory, `.xtc` in our case
- `-s` &mdash; (mandatory) `.tpr` file has default atom groups in case you didn't prepare `.ndx` file and other information, e.g., mass, which is necessary to calculate mass density, etc.
- `-n` &mdash; `.ndx` file
- `-o` &mdash; output density file in `.xvg` format
- `-b` &mdash; simulation time to start analysis (useful when you want to analyze only a fragment of your simulation)
- `-e` &mdash; simulation time to end analysis (useful when you want to analyze only a fragment of your simulation)
- `-dens` &mdash; which density to calculate: `mass`, `number`, `charge`, or `electron`
- `-sl` &mdash; divide the box in this number of slices (defines the resolution of the data)
- `-d` &mdash; box dimension across which the density is calculated: `x`, `y`, `z`; default is `z`
- `-ng` &mdash; you can calculate more than one density enough; `-ng` specifies number of atom groups to calculate density for; by default, equals 1
- there are other options that we will not cover today; check `gmx density -h` or https://manual.gromacs.org/current/onlinehelp/gmx-density.html yourself

In [ ]:
input_command = """OW 
"""
!echo "$input_command" | gmx density -f ../trajectories/long_pure_water.xtc -n pure_water.ndx -s ../trajectories/long_pure_water.tpr -dens number -sl 300 -o dens_OW_z_pure_water.xvg

In [ ]:
dens_OW_z_pure_water = np.loadtxt('dens_OW_z_pure_water.xvg', comments=['#', '@'])
plt.plot(dens_OW_z_pure_water[:, 0], dens_OW_z_pure_water[:, 1], 'b-')
plt.legend(['water'])
plt.xlabel('z (nm)')
plt.ylabel('Density ($\mathrm{nm^{-3}}$)')
plt.ylim([25, 40])
plt.show()

### Not big surprise that water is distributed in the box of water homogenously :)

- Time to check one of the mixtures.

In [ ]:
input_command = """a OW
a OA
q
"""
!echo "$input_command" | gmx make_ndx -f ../trajectories/long_mix_w1_m1.gro -o mix_w1_m1.ndx

### Observation: GROMACS can behave stupid (see above)

- It sees carbons so it suggests to unite carbons and related atoms into `Protein` group, etc.
- I warned you before.
- Of course, it does not affect any simulation results, only post-processing. Be careful.

In [ ]:
input_command = """OW
OA
"""
!echo "$input_command" | gmx density -f ../trajectories/long_mix_w1_m1.xtc -n mix_w1_m1.ndx -s ../trajectories/long_mix_w1_m1.tpr -dens number -sl 300 -ng 2 -o dens_OW_OA_z_mix_w1_m1.xvg

In [ ]:
!head ../trajectories/long_pure_methanol.gro

In [ ]:
dens_OW_OA_z_mix_w1_m1 = np.loadtxt('dens_OW_OA_z_mix_w1_m1.xvg', comments=['#', '@'])
plt.plot(dens_OW_OA_z_mix_w1_m1[:, 0], dens_OW_OA_z_mix_w1_m1[:, 1], 'b-')
plt.plot(dens_OW_OA_z_mix_w1_m1[:, 0], dens_OW_OA_z_mix_w1_m1[:, 2], 'r-')
plt.legend(['water', 
            'methanol'])
plt.xlabel('z (nm)')
plt.ylabel('Density ($\mathrm{nm^{-3}}$)')
plt.ylim([5, 15])
plt.show()

### Looks fine

### TASK

- Check mixability of other two mixtures: mix_w1_m3 and mix_w3_m1.

## Radial distribution functions

- Corresponding atom groups typically should be created using `gmx index`.
- Then `gmx rdf` is used to calculate RDFs.

### `gmx rdf` 

- `-f` &mdash; input trajectory, `.xtc` in our case
- `-s` &mdash; (usually optional) `.tpr` file has default atom groups in case you didn't prepare `.ndx` file
- `-n` &mdash; `.ndx` file
- `-o` &mdash; output RDF file in `.xvg` format
- `-b` &mdash; simulation time to start analysis (useful when you want to analyze only a fragment of your simulation)
- `-e` &mdash; simulation time to end analysis (useful when you want to analyze only a fragment of your simulation)
- `-bin` &mdash; binning of the distance (defines the resolution of the data); default &mdash; 0.002 nm; usually enough, depends on the number of species of interest in the system and how long the trajectory is (more specifically, how many simulation frames you have in your trajectory)
- you can specify groups to analyze either directly in the command using `-ref` and `-sel` or select them interactively
- there are other options that we will not cover today; check `gmx rdf -h` or https://manual.gromacs.org/current/onlinehelp/gmx-rdf.html yourself

## RDFs in our systems

- Remember this RDF of water in the lecture? Let's reproduce it.
- We do not need to create a new group in `.ndx` file so far, we already have water oxygens.

In [ ]:
!gmx rdf -f ../trajectories/long_pure_water.xtc -n pure_water.ndx -o rdf_OW_OW_pure_water.xvg -ref OW -sel OW

In [ ]:
rdf_OW_OW_pure_water = np.loadtxt('rdf_OW_OW_pure_water.xvg', comments=['#', '@'])
plt.plot(rdf_OW_OW_pure_water[:, 0], rdf_OW_OW_pure_water[:, 1], 'b-')
plt.legend(['RDF OW-OW'])
plt.xlabel('r (nm)')
plt.ylabel('g (r)')
plt.show()

### TASK

- Calculate RDFs for water hydrogens around water oxygens
- Hint: water hydrogens are named `HW1` and `HW2`
- To rename a group in index file, use command like `name 3 TEST`, where `3` is the index of the group and `TEST` is the new name of the group

### RDFs of methanol

In [ ]:
input_command = """a OA
q
"""
!echo "$input_command" | gmx make_ndx -f ../trajectories/long_pure_methanol.gro -o pure_methanol.ndx

In [ ]:
!gmx rdf -f ../trajectories/long_pure_methanol.xtc -n pure_methanol.ndx -o rdf_OA_OA_pure_methanol.xvg -ref OA -sel OA

In [ ]:
rdf_OA_OA_pure_methanol = np.loadtxt('rdf_OA_OA_pure_methanol.xvg', comments=['#', '@'])
plt.plot(rdf_OA_OA_pure_methanol[:, 0], rdf_OA_OA_pure_methanol[:, 1], 'r-')
plt.legend(['RDF OA-OA'])
plt.xlabel('r (nm)')
plt.ylabel('g (r)')
plt.show()

### Let's plot together pure water and pure methanol

In [ ]:
plt.plot(rdf_OW_OW_pure_water[:, 0], rdf_OW_OW_pure_water[:, 1], 'b-')
plt.plot(rdf_OA_OA_pure_methanol[:, 0], rdf_OA_OA_pure_methanol[:, 1], 'r-')
plt.legend(['RDF OW-OW', 
            'RDF OA-OA'])
plt.xlabel('r (nm)')
plt.ylabel('g (r)')
plt.show()

### Discussion

- What do we see?

### TASK

- Calculate RDFs for water-water, methanol-methanol, and water-methanol from any mixture of your choice.
- Again discussion.

## Simulated densities can be compared to the experimental data

- A lot of thermodynamic measurements on solutions like this have been performed in 1960-1970s.
- Water/methanol mixtures are not en exception.
    - McGlashan and Williamson, J. Chem. Eng. Data 1976, 21, 2, 196–199 https://doi.org/10.1021/je60069a019

<img src='../images/water_methanol_densities_exp1.png' width=600 height=450 alt='exp_densities_1'>

### Does anybody notice anything strange?

- Hint: even published articles have mistakes/typos. More often than you think.

### Question

- The first column is the mole fraction. What is that?

### Preparing files to plot

- This step is not necessary, by the way, you can directly plot the data typed into a Jupyter notebook.

In [ ]:
exp_densities_1 = """# mole fraction, density [g/ml]
0.04085 0.98472
0.06168 0.97919
0.11445 0.96649
0.15546 0.95730
0.19739 0.94796
0.24867 0.93658
0.34382 0.91534
0.49446 0.88242
0.61267 0.85790
0.69241 0.84210
0.78454 0.82458
0.89229 0.80510
1.00000 0.78663
"""
with open('exp_densities_1.xvg', 'w') as outFile:
    outFile.write(exp_densities_1)

In [ ]:
!cat exp_densities_1.xvg

In [ ]:
# UNCOMMENT
# sim_densities = """# mole fraction, density [g/ml]
# 0.00 PUT_THE_VALUE_FROM_BEFORE
# 0.25 PUT_THE_VALUE_FROM_BEFORE
# 0.50 PUT_THE_VALUE_FROM_BEFORE
# 0.75 PUT_THE_VALUE_FROM_BEFORE
# 1.00 PUT_THE_VALUE_FROM_BEFORE
# """
# with open('sim_densities.xvg', 'w') as outFile:
#     outFile.write(sim_densities)

In [ ]:
!cat sim_densities.xvg

In [ ]:
exp_densities_1 = np.loadtxt('exp_densities_1.xvg', comments=['#'])
exp_densities_1[:, 1] *= 1000 # we multiple by 1000 to correct for unigs
sim_densities = np.loadtxt('sim_densities.xvg', comments=['#'])
plt.plot(exp_densities_1[:, 0], exp_densities_1[:, 1], 'r*-')
plt.plot(sim_densities[:, 0], sim_densities[:, 1], 'b*-')
plt.legend(['experimental density #1', 
            'simulation density'])
plt.xlabel('Mole fraction of methanol')
plt.ylabel('Density (kg/$\mathrm{m^3}$)')
plt.show()

### Not ideal but good to go :)

### Another set of experimental data, why not

- Martens et al., Chem. Ing. Tech. 2020, 92, 10, 1474–1481 https://doi.org/10.1002/cite.202000058

<img src='../images/water_methanol_densities_exp2.png' width=600 height=450 alt='exp_densities_2'>

### We should convert weight contribution (wt %) to mole fraction, a bit of thermodynamics

- Let's assume we have 100 g of water/methanol mixture
- 10.09% of methanol means we have 10.09 g of methanol and 89.91 g of water
- Molar mass of methanol = 32 g/mol
- Molar mass of water = 18 g/mol
- Let's calculate the number of moles of methanol, total number of moles, and mole fraction of methanol

In [ ]:
m_meth = 10.09
n_meth = m_meth / 32
n_water = (100 - m_meth) / 18
n_total = n_meth + n_water
n_meth_fract = round(n_meth / n_total, 3)
print(n_meth_fract)

### Be smart

- Optimize the code to calculate the mole fraction for all the values from the picture

In [ ]:
m_meth_list = [0, 10.09, 19.92, 29.83, 40.05, 50.00, 59.91, 70.23, 80.02, 90.03, 100]
for m_meth in m_meth_list:
    n_meth = m_meth / 32
    n_water = (100 - m_meth) / 18
    n_total = n_meth + n_water
    n_meth_fract = round(n_meth / n_total, 3)
    print(n_meth_fract)

### Combining with writing a file with the experimental data

- The code combines loops in Python, related simple calculations, and writing into the file.
- Can we simplity it even more?

In [ ]:
exp_densities_values_2 = [0.9970, 0.9797, 0.9646, 0.9489, 0.9311, 0.9117, 0.8905, 0.8666, 0.8421, 0.8151, 0.7866]
m_meth_list = [0, 10.09, 19.92, 29.83, 40.05, 50.00, 59.91, 70.23, 80.02, 90.03, 100]

with open('exp_densities_2.xvg', 'w') as outFile:
    outFile.write('# mole fraction, density [g/ml]\n')
    for m_meth, density in zip(m_meth_list, exp_densities_values_2):
        n_meth = m_meth / 32
        n_water = (100 - m_meth) / 18
        n_total = n_meth + n_water
        n_meth_fract = round(n_meth / n_total, 3)
        outFile.write(f"{n_meth_fract}\t{density}\n")

In [ ]:
!cat exp_densities_2.xvg

In [ ]:
exp_densities_2 = np.loadtxt('exp_densities_2.xvg', comments=['#'])
exp_densities_2[:, 1] *= 1000
plt.plot(exp_densities_1[:, 0], exp_densities_1[:, 1], 'r*-')
plt.plot(exp_densities_2[:, 0], exp_densities_2[:, 1], 'm*-')
plt.plot(sim_densities[:, 0], sim_densities[:, 1], 'b*-')
plt.legend(['experimental density #1', 
            'experimental density #2', 
            'simulation density #2'])
plt.xlabel('Mole fraction of methanol')
plt.ylabel('Density (kg/$\mathrm{m^3}$)')
plt.show()

### Experiments agree with each other :) simulations are just a bit off

## Self-diffusion coefficient

- Let's calculate the self-diffusion coefficient of water in pure water solution and in a mixture with methanol.
- Although it looks like a simple property to calculate, there are many details to be aware of.
    - https://livecomsjournal.org/index.php/livecoms/article/view/v1i1e6324 - beautiful "practices" that elaborate on related methods and technical details

### gmx msd

- `gmx msd` is used to calculate the mean square displacement; from its linear fit, the self-diffusion coefficient can be estimated using the Einstein relation
- `-f` &mdash; input trajectory, `.xtc` in our case
- `-s` &mdash; (usually optional) `.tpr` file has default atom groups in case you didn't prepare `.ndx` file
- `-n` &mdash; `.ndx` file
- `-b` &mdash; simulation time to start analysis (useful when you want to analyze only a fragment of your simulation)
- `-e` &mdash; simulation time to end analysis (useful when you want to analyze only a fragment of your simulation)
- `-beginfit` &mdash; simulation time to start fitting (by default, first 10% are not included into the fitting)
- `-endfit` &mdash; simulation time to end fitting (by default, last 10% are not included into the fitting)
- `-o` &mdash; output file in `.xvg` format
- there are other options that we will not cover today; check `gmx msd -h` or https://manual.gromacs.org/current/onlinehelp/gmx-msd.html yourself

## Box-size depencence on the diffusion

- Due to periodic boundary conditions (next week), dynamic properties are affected by the box size of your simulation.
- Solution: the data should be corrected and reported as for the infinite box size.
- The solution is either analytical or extrapolated from several box sizes modeled.

### Analytical solution

- Correction by Yeh and Hummer (J. Phys. Chem. B 2004, 108, 40, 15873–15879)
    - https://doi.org/10.1021/jp0477147
- $D_{0} = D_{PBC} + 2.837297 k_{b} T/(6 \pi \eta L)$
    - $D_{0}$ &mdash; self-diffusion coefficient at infinite box size
    - $D_{PBC}$ &mdash; self-diffusion coefficient calculated from a simulation
    - $k_{b}$ &mdash; the Boltzmann constant
    - $T$ &mdash; absolute temperature
    - $\pi$ &mdash; I hope I don't need to tell you :)
    - $\eta$ &mdash; shear viscosity of the solvent (you should know this value!)
    - $L$ &mdash; box length

### Extrapolation

- If you have two simulations, you can calculate the data, and plot the line.
- As you should know, two points are enough to plot a line...
    - ... but three and more points are usually better

<img src='../images/extrapolation.png' width=600 height=450 alt='extrapolation'>

### Self-diffusion coefficient from our simulations

- Water oxygens are enough.
- Essentially identical to the MSD of the center of mass of a water molecule.
- This approximation is not always the case for other molecules (rather opposite), particularly big ones.

In [ ]:
!gmx msd -f ../trajectories/long_pure_water.xtc -n pure_water.ndx -o dif_OW_pure_water.xvg -sel OW

In [ ]:
dif_OW_pure_water = np.loadtxt('dif_OW_pure_water.xvg', comments=['#', '@'])
plt.plot(dif_OW_pure_water[:, 0], dif_OW_pure_water[:, 1], 'r-')
plt.legend(['MSD OW'])
plt.xlim([1000, 9000])
plt.xlabel('tau (ps)')
plt.ylabel('MSD ($\mathrm{nm^2}$)')
plt.show()

### Note that the header of output `.xvg` file has already self-diffusion coefficient calculated for you

### Note that error bars are also given

- Calculated as the difference of the diffusion coefficients obtained from fits over the two halves of the fit interval.

In [ ]:
!head -n 40 dif_OW_pure_water.xvg

### How to extract this line nicely using Python?

In [ ]:
with open(f"dif_OW_pure_water.xvg", 'r') as file:
    content = file.read()
    match = re.search(r'^(.*cm\^2/s.*)$', content, re.MULTILINE)
    if match:
        print(match.group(1))
    else:
        print("Something went wrong!")

### What's about bigger boxes we have?
- Below we are creating `.ndx` files for larger systems and also calculate the self-diffusion.
- Be smart. Use python to optimize this step-by-step procedure.

In [ ]:
# input_command = """a OW
# q
# """
# !echo "$input_command" | gmx make_ndx -f ../trajectories/long_pure_water_bg.gro -o pure_water_bg.ndx

In [ ]:
# input_command = """a OW
# q
# """
# !echo "$input_command" | gmx make_ndx -f ../trajectories/long_pure_water_lg.gro -o pure_water_lg.ndx

In [ ]:
# input_command = """a OW
# q
# """
# !echo "$input_command" | gmx make_ndx -f ../trajectories/long_pure_water_hg.gro -o pure_water_hg.ndx

In [ ]:
# !gmx msd -f ../trajectories/long_pure_water_bg.xtc -n pure_water_bg.ndx -o dif_OW_pure_water_bg.xvg -sel OW

In [ ]:
# !gmx msd -f ../trajectories/long_pure_water_lg.xtc -n pure_water_lg.ndx -o dif_OW_pure_water_lg.xvg -sel OW

In [ ]:
# !gmx msd -f ../trajectories/long_pure_water_hg.xtc -n pure_water_hg.ndx -o dif_OW_pure_water_hg.xvg -sel OW

In [ ]:
dif_OW_pure_water_bg = np.loadtxt('dif_OW_pure_water_bg.xvg', comments=['#', '@'])
dif_OW_pure_water_lg = np.loadtxt('dif_OW_pure_water_lg.xvg', comments=['#', '@'])
dif_OW_pure_water_hg = np.loadtxt('dif_OW_pure_water_hg.xvg', comments=['#', '@'])
plt.plot(dif_OW_pure_water[:, 0], dif_OW_pure_water[:, 1], 'r-')
plt.plot(dif_OW_pure_water_bg[:, 0], dif_OW_pure_water_bg[:, 1], 'b-')
plt.plot(dif_OW_pure_water_lg[:, 0], dif_OW_pure_water_lg[:, 1], 'g-')
plt.plot(dif_OW_pure_water_hg[:, 0], dif_OW_pure_water_hg[:, 1], 'y-')
plt.legend(['MSD OW', 
            'MSD OW big box',
            'MSD OW huge box',
            'MSD OW large box'])
plt.xlim([1000, 9000])
plt.xlabel('tau (ps)')
plt.ylabel('MSD ($\mathrm{nm^2}$)')
plt.show()

In [ ]:
systems = ['pure_water', 'pure_water_bg', 'pure_water_lg', 'pure_water_hg']
for system in systems:
    with open(f"dif_OW_{system}.xvg", 'r') as file:
        content = file.read()
        match = re.search(r'^(.*cm\^2/s.*)$', content, re.MULTILINE)
        if match:
            print(match.group(1))
        else:
            print("Something went wrong!")

### Recall how much water we have in each system

- Original: 1000 waters
- bg: 2000 waters
- lg: 3000 waters
- hg: 9000 waters

### Dynamical properties need longer simulations to converge

- Original, big, and large boxes give very similar results although they should be somewhat different (larger box, higher self-diffusion)
- For the biggest system, the difference is obvious

### Analytical correction

- Let's apply analytical correction to two systems - pure_water_bg (small error bar) and pure_water_hg (much larger system and higher self-diffusion coefficient)
- $D_{0} = D_{PBC} + 2.837297 k_{b} T/(6 \pi \eta L)$
    - $D_{0}$ &mdash; self-diffusion coefficient at infinite box size
    - $D_{PBC}$ &mdash; self-diffusion coefficient calculated from a simulation
    - $k_{b}$ &mdash; the Boltzmann constant
    - $T$ &mdash; absolute temperature
    - $\pi$ &mdash; I hope I don't need to tell you :)
    - $\eta$ &mdash; shear viscosity of the solvent (you should know this value in advance!)
    - $L$ &mdash; box length

### What do we have and shall we know?

- $D_{0}$ &mdash; we want to calculate this
- $D_{PBC}$ &mdash; let's say, 2.1013 1e-5 cm$^2$/s
- $k_{b}$ &mdash; the Boltzmann constant, 1.38 1e-23 J/K
- $T$ &mdash; absolute temperature, 298.15 K
- $\pi$ &mdash; I still hope I don't need to tell you :)
- $\eta$ &mdash; shear viscosity of water (to be more precise, of the water model we use, TIP45/2005), 0.855 mPa·s 
- $L$ &mdash; box length, we calculate it below

In [ ]:
!echo 'Box-Z' | gmx energy -f ../trajectories/long_pure_water_bg.edr -o box_z_pure_water_bg.xvg

In [ ]:
# diff_OW_inf_bg = 2.1000 + (2.837297 * 1.38 * 10**(-23) * 298.15) / (6 * math.pi * 0.855 * PUT_THE_VALUE_FROM_BEFORE * 10**(-21))
# print(round(diff_OW_inf_bg, 4))

In [ ]:
!echo 'Box-Z' | gmx energy -f ../trajectories/long_pure_water_hg.edr -o box_z_pure_water_hg.xvg

In [ ]:
# diff_OW_inf_hg = 2.1851 + (2.837297 * 1.38 * 10**(-23) * 298.15) / (6 * math.pi * 0.855 * PUT_THE_VALUE_FROM_BEFORE * 10**(-21))
# print(round(diff_OW_inf_hg, 4))

### Linear extrapolation

In [ ]:
# invert_box_size_data = np.array([1/PUT_BOX_SIZE_HERE, 1/PUT_BOX_SIZE_HERE])
# self_diff_data = np.array([PUT_SELF_DIFF_HERE, PUT_SELF_DIFF_HERE])
# coefficients = np.polyfit(invert_box_size_data, self_diff_data, 1)
# p = np.poly1d(coefficients)
# self_diff_inf = p(0)
# print(self_diff_inf)

### Correction clearly helps

- Longer simulations are required to converge the data.
- 10 ns is actually nothing.

### TASK

- Basic error estimation from one simulation.
- Calculate the self-diffusion coefficient for the biggest system separately from the first and second halves of the system.
- Conclusions?

### TASK

- Would be the self-diffusion coefficient different for methanol carbon and methanol oxygen?
- Calculate them from one of the systems and compare.
- Conclusion?

### TASK

- Calculate the self-diffusion of water from water/methanol mixtures (three box sizes).
- Extrapolate the data to the infinite box size.
- How does the self-diffusion of water in the mixture compare to pure water?

### Are structural properties box size-dependent? No (unless you are doing something wrong)

- Structural properties are less-affected by long-range interactions.
- More about long-range interactions and interactions in general &mdash; next week.
- They also need less time to converge.

### TASK

- Calculate densities and RDFs from water simulations with different box sizes and compare them.

## More about errors

- Block averaging is a technique that can help reduce the autocorrelation of data, providing more accurate error estimates.
- If you have a measure as a function time, block averaging is an excellent tool the estimate the error.
- `gmx analyze` has block averaging implemented.
    - Give it `.xvg` file or prepare a kinda `.xvg` yourself.
    - The idea is simple: the file should have the first column as time and other columns are the data measured as a function of time.
- We already have temperature and density data like that.

### gmx analyze

- `gmx analyze` is a versatile tool to analyze data sets.
- `-f` &mdash; input `.xvg` file
- `-ee` &mdash; output `.xvg` file with the error estimate
- **there are a lot of other options** that we will not cover today; check `gmx analyze -h` or https://manual.gromacs.org/current/onlinehelp/gmx-analyze.html yourself

In [ ]:
!gmx analyze -f temp_pure_water.xvg -ee temp_error_pure_water.xvg

### TASK

- Calculate errors for other data we already have.

## MDAnalysis: first try

- Analyses that are already tested with GROMACS:
    - https://userguide.mdanalysis.org/stable/examples/analysis/structure/average_rdf.html
- Let's try RDF.

In [ ]:
import MDAnalysis as mda
from MDAnalysis.analysis import rdf

u = mda.Universe('../trajectories/long_pure_water.gro', '../trajectories/long_pure_water.xtc')

OW = u.select_atoms('name OW')
HW = u.select_atoms('name HW1 HW2')
rdf_OW_OW_pure_water_mda = rdf.InterRDF(OW, HW, bins=100)
rdf_OW_OW_pure_water_mda.run()

In [ ]:
plt.plot(rdf_OW_OW_pure_water_mda.results.bins, rdf_OW_OW_pure_water_mda.results.rdf, 'b-')
plt.legend(['RDF OW-HW MDA'])
plt.xlabel('r (A)')
plt.ylabel('g (r)')
plt.show()

### QUESTIONS AND TASKS

- What is weird in the profile above?
- Plot it together with the one from GROMACS tools. What is the difference? Elaborate on that.

### FINAL MORE INDEPENDENT TAST
- Calculate density of whatever for any system we have and compare it to the one calculates using GROMACS tools.
- https://userguide.mdanalysis.org/stable/examples/analysis/volumetric/linear_density.html
- The match should be ideal.